# Tome C1, Chapter 5: Graphs (Python companion)

Exercises first, solutions at the end. Every proof of the chapter is an algorithm, and here we run them: a bipartiteness test that returns an odd cycle when it says no, Hierholzer's Euler tours, de Bruijn sequences built two ways and counted by brute force, Dirac's rotation trick, Rédei's insertion sort on tournaments, Menger's theorem as a max-flow computation, ear decompositions and Robbins's one-way streets.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, random, math, time
from collections import defaultdict, deque, Counter
import numpy as np
import matplotlib.pyplot as plt
random.seed(5)
def random_graph(n, p, rng=random):
    adj = {v: set() for v in range(n)}
    for u, v in itertools.combinations(range(n), 2):
        if rng.random() < p:
            adj[u].add(v); adj[v].add(u)
    return adj
def connected(adj, removed=()):
    vs = [v for v in adj if v not in removed]
    if not vs: return True
    seen = {vs[0]}; st = [vs[0]]
    while st:
        u = st.pop()
        for w in adj[u]:
            if w not in seen and w not in removed:
                seen.add(w); st.append(w)
    return len(seen) == len(vs)

## Exercises

**P5.1 (★) Two colours or an odd cycle.** Write a breadth-first search that either 2-colours a graph or returns an odd cycle, as in Kőnig's theorem (Theorem 5.2). The answer must be a *certificate*: check the colouring or the cycle independently, on thousands of random graphs.

**P5.2 (★★) Hierholzer.** Implement Hierholzer's algorithm for multigraphs and directed multigraphs (Theorem 5.4), in linear time with a stack. Test it on random Eulerian multigraphs, check Königsberg, lay the dominoes in a ring (Exercise 5.2), and partition a graph with $2k$ odd vertices into $k$ trails (Exercise 5.1).

**P5.3 (★★) De Bruijn sequences.** Build de Bruijn sequences (a) as Euler tours of $D(q,n)$ (Theorem 5.5) and (b) by concatenating Lyndon words (the remark after it). Check both. Count the de Bruijn sequences by brute force and compare with $(q!)^{q^{n-1}}/q^n$. Finally, perform the card trick of Exercise 5.3.

**P5.4 (★★) Dirac's rotation trick.** Turn the proof of Theorem 5.6 into an algorithm that finds a Hamiltonian cycle when every degree is at least $n/2$, and run it on graphs with hundreds of vertices. With an exact (exponential) Hamiltonicity test, check that the graphs of Exercise 5.6 have none.

**P5.5 (★★) Tournaments.** Rank random tournaments by Rédei's insertion with binary search (Theorem 5.7), counting the games looked up. Count Hamiltonian paths exactly by dynamic programming and check that the count is always odd. Check Camion's theorem (Exercise 5.4) and Landau's kings (Exercise 5.5).

**P5.6 (★★★) Menger by flows.** Compute the maximum number of disjoint $A$–$B$ paths by augmenting paths in the split graph (each vertex becomes an arc of capacity $1$). Compare with the minimum separator found by brute force (Theorem 5.8), and compute the connectivity of some famous graphs.

**P5.7 (★★) Ears, one-way streets and Gray codes.** Compute an ear decomposition of a 2-connected graph by depth-first search (Theorem 5.10). Orient a bridgeless graph by DFS (Exercise 5.9) and check strong connectivity (Theorem 5.11). Check the reflected Gray code (Exercise 5.7).

## Solutions

### P5.1
**How to find it.** Colour by the parity of the BFS level. If an edge $uv$ joins two vertices of the same level parity, the two BFS-tree paths from $u$ and $v$ up to their lowest common ancestor, plus $uv$, form a closed walk of odd length; since both tree paths go up one level per step and meet at the ancestor, the result is in fact a cycle.

In [ ]:
def bipartite_or_odd_cycle(adj):
    colour, parent, depth = {}, {}, {}
    for s in adj:
        if s in colour: continue
        colour[s], parent[s], depth[s] = 0, None, 0
        q = deque([s])
        while q:
            u = q.popleft()
            for w in adj[u]:
                if w not in colour:
                    colour[w], parent[w], depth[w] = 1 - colour[u], u, depth[u] + 1
                    q.append(w)
                elif colour[w] == colour[u]:
                    a, b, left, right = u, w, [u], [w]
                    while a != b:                       # climb to the lowest common ancestor
                        if depth[a] >= depth[b]: a = parent[a]; left.append(a)
                        else: b = parent[b]; right.append(b)
                    return False, left + right[-2::-1]  # u ... lca ... w
    return True, colour

def check(adj, res):
    ok, cert = res
    if ok:
        return all(cert[u] != cert[v] for u in adj for v in adj[u])
    c = cert
    return (len(c) % 2 == 1 and len(set(c)) == len(c)
            and all(c[(i + 1) % len(c)] in adj[c[i]] for i in range(len(c))))

stats = Counter()
for trial in range(3000):
    n = random.randint(1, 25); adj = random_graph(n, random.choice([0.05, 0.1, 0.2]))
    res = bipartite_or_odd_cycle(adj)
    assert check(adj, res)
    stats['bipartite' if res[0] else 'odd cycle'] += 1
print(stats, '- every answer certified')
pet = {i: set() for i in range(10)}
for i in range(5):
    for u, v in [(i, (i + 1) % 5), (5 + i, 5 + (i + 2) % 5), (i, 5 + i)]:
        pet[u].add(v); pet[v].add(u)
print('Petersen graph:', bipartite_or_odd_cycle(pet))

### P5.2
**How to find it.** Hierholzer with a stack: walk along unused edges, pushing vertices; when stuck, pop the vertex onto the tour. Each edge is used once and each vertex pushed once per edge, so the time is linear. For multigraphs, edges carry identifiers and each vertex keeps a pointer into its list of incident edges.

In [ ]:
def euler_tour(n, edges, directed=False, start=None, with_ids=False):
    # edges: list of (u, v); returns the list of vertices of a closed tour, or None
    inc = [[] for _ in range(n)]
    for i, (u, v) in enumerate(edges):
        inc[u].append((v, i))
        if not directed: inc[v].append((u, i))
    if directed:
        outd, ind = Counter(u for u, v in edges), Counter(v for u, v in edges)
        if any(outd[v] != ind[v] for v in range(n)): return None
    elif any(len(inc[v]) % 2 for v in range(n)):   # a loop counts twice
        return None
    used, ptr = [False] * len(edges), [0] * n
    if start is None: start = edges[0][0] if edges else 0
    stack, tour = [(start, None)], []
    while stack:
        u = stack[-1][0]
        while ptr[u] < len(inc[u]) and used[inc[u][ptr[u]][1]]: ptr[u] += 1
        if ptr[u] == len(inc[u]):
            tour.append(stack.pop())
        else:
            v, i = inc[u][ptr[u]]; used[i] = True; stack.append((v, i))
    if len(tour) != len(edges) + 1: return None          # not connected
    tour.reverse()
    verts = [v for v, i in tour]
    return (verts, [i for v, i in tour[1:]]) if with_ids else verts

def is_tour(tour, edges, directed=False):
    key = (lambda a, b: (a, b)) if directed else (lambda a, b: tuple(sorted((a, b))))
    return tour[0] == tour[-1] and Counter(key(a, b) for a, b in zip(tour, tour[1:])) == Counter(key(*e) for e in edges)

# random Eulerian multigraphs: unions of random closed walks
for trial in range(500):
    n = random.randint(2, 12); edges = []
    for _ in range(random.randint(1, 4)):
        w = [0] + [random.randrange(n) for _ in range(random.randint(1, 8))] + [0]
        edges += list(zip(w, w[1:]))          # closed walks through 0: connected, Eulerian
    start = 0
    for directed in (False, True):
        t = euler_tour(n, edges, directed, start)
        assert t is not None and is_tour(t, edges, directed)
print('500 random multigraphs, undirected and directed: tours verified')
K = ['N', 'I', 'S', 'E']
kon = [(0, 1), (0, 1), (2, 1), (2, 1), (0, 3), (2, 3), (1, 3)]
print('Koenigsberg degrees:', {K[v]: d for v, d in Counter(x for e in kon for x in e).items()}, '-> tour:', euler_tour(4, kon))
for top in (6, 7):
    dom = [(a, b) for a in range(top + 1) for b in range(a, top + 1)]
    t = euler_tour(top + 1, dom)
    print(f'dominoes 0..{top} ({len(dom)} tiles):', 'ring ' + ' '.join(f'[{a}|{b}]' for a, b in zip(t, t[1:])) if t else 'impossible (degrees are odd)')

In [ ]:
# Exercise 5.1: k trails. Pair the odd vertices with k fake edges, take a tour, cut at the fake edges.
def min_trails(n, edges):
    odd = [v for v, d in Counter(x for e in edges for x in e).items() if d % 2]
    fake = [(odd[i], odd[i + 1]) for i in range(0, len(odd), 2)]
    if not fake: return [euler_tour(n, edges)], 0
    verts, ids = euler_tour(n, edges + fake, start=odd[0], with_ids=True)
    j = ids.index(len(edges))                  # rotate the closed tour to start just after a fake edge
    verts = verts[j + 1:-1] + verts[:j + 1]; ids = ids[j + 1:] + ids[:j + 1]
    trails, cur = [], [verts[0]]
    for v, i in zip(verts[1:] + [verts[0]], ids):
        if i >= len(edges):
            trails.append(cur); cur = [v]
        else:
            cur.append(v)
    return trails, len(fake)
random.seed(1)
for trial in range(300):
    n = random.randint(3, 10)
    adj = random_graph(n, 0.5)
    if not connected(adj): continue
    edges = [(u, v) for u in adj for v in adj[u] if u < v]
    if not edges: continue
    trails, k = min_trails(n, edges)
    used = Counter(tuple(sorted(x)) for tr in trails for x in zip(tr, tr[1:]))
    assert used == Counter(tuple(sorted(e)) for e in edges) and len(trails) == max(k, 1)
print('Exercise 5.1 checked: k trails cover every edge exactly once')

### P5.3
**How to find it.** (a) The vertices of $D(q,n)$ are the words of length $n-1$, the arcs the words of length $n$; an Euler tour read by first letters is the sequence. (b) The Fredricksen–Kessler–Maiorana algorithm generates the Lyndon words in lexicographic order in constant amortised time; concatenating those whose length divides $n$ gives the lexicographically smallest de Bruijn sequence. For the count, generate all words of length $q^n$, keep those whose $q^n$ cyclic windows are distinct, and divide by $q^n$ (each cyclic sequence has $q^n$ distinct rotations, since the windows are distinct).

In [ ]:
def de_bruijn_euler(q, n):
    if n == 1: return list(range(q))
    verts = list(itertools.product(range(q), repeat=n - 1)); idx = {v: i for i, v in enumerate(verts)}
    edges = [(idx[w[:-1]], idx[w[1:]]) for w in itertools.product(range(q), repeat=n)]
    t = euler_tour(len(verts), edges, directed=True, start=0)
    return [verts[v][0] for v in t[:-1]]

def de_bruijn_lyndon(q, n):
    # Fredricksen-Kessler-Maiorana (Duval's generation of Lyndon words)
    seq, w = [], [-1]
    while w:
        w[-1] += 1
        if n % len(w) == 0: seq += w
        m = len(w)
        while len(w) < n: w.append(w[len(w) - m])
        while w and w[-1] == q - 1: w.pop()
    return seq

def is_de_bruijn(s, q, n):
    L = len(s); ss = s + s[:n - 1]
    return L == q ** n and len({tuple(ss[i:i + n]) for i in range(L)}) == L

for q, n in [(2, 1), (2, 3), (2, 4), (2, 10), (3, 2), (3, 5), (4, 4), (10, 4)]:
    a, b = de_bruijn_euler(q, n), de_bruijn_lyndon(q, n)
    assert is_de_bruijn(a, q, n) and is_de_bruijn(b, q, n)
print('B(2,4) by Lyndon words:', ''.join(map(str, de_bruijn_lyndon(2, 4))))
s = de_bruijn_lyndon(10, 4)
print('the safe: one sequence of', len(s) + 3, 'presses tries all 10^4 codes; starts', ''.join(map(str, s[:20])), '...')

In [ ]:
def count_brute(q, n):
    L = q ** n; c = 0
    for s in itertools.product(range(q), repeat=L):
        if s[:n] == (0,) * n and is_de_bruijn(list(s), q, n):  # fix the rotation starting with 0^n
            c += 1
    return c
for q, n in [(2, 1), (2, 2), (2, 3), (2, 4), (3, 1), (3, 2), (4, 1)]:
    t0 = time.time()
    formula = math.factorial(q) ** (q ** (n - 1)) // q ** n
    print(f'q={q}, n={n}: brute force {count_brute(q, n):>3}, formula (q!)^(q^(n-1))/q^n = {formula}')

Fixing the rotation that starts with $0^n$ picks exactly one representative per cyclic sequence, since $0^n$ appears exactly once. For $q=2$ the formula is Flye Sainte-Marie's $2^{2^{n-1}-n}$; the $16$ of $(2,4)$ answers the last question of Exercise 5.3. Now the card trick with $32$ cards.

In [ ]:
s = de_bruijn_lyndon(2, 5)
suits = [f'{v}{c}' for c in 'SHDC' for v in ['A', '2', '3', '4', '5', '6', '7', '8']]
reds = [c for c in suits if c[-1] in 'HD']; blacks = [c for c in suits if c[-1] in 'SC']
deck = [reds.pop() if bit else blacks.pop() for bit in s]       # red = 1
where = {tuple((s + s)[i:i + 5]): i for i in range(32)}
cut = random.randrange(32)
colours = tuple((s + s)[cut:cut + 5])
print('colours called out:', ''.join('R' if c else 'B' for c in colours))
i = where[colours]
print('the magician names:', (deck + deck)[i:i + 5], ' actual:', (deck + deck)[cut:cut + 5])

### P5.4
**How to find it.** Keep a path; extend it while an endpoint has a neighbour outside. When stuck, the proof finds $i$ with $v_0v_{i+1}$ and $v_iv_k$ edges and closes a cycle; if the cycle misses a vertex $u$ adjacent to some $v_j$ on it, open the cycle at $v_j$ and prepend $u$: a longer path. Each round adds a vertex, so there are at most $n$ rounds of $O(n^2)$ work. For the exact test, the Held–Karp dynamic programme over subsets.

In [ ]:
def dirac_cycle(adj):
    n = len(adj); path = [0]; on = {0}
    while True:
        extended = True
        while extended:                           # greedy extension at both ends
            extended = False
            for end in (0, -1):
                for w in adj[path[end]]:
                    if w not in on:
                        (path.insert(0, w) if end == 0 else path.append(w)); on.add(w); extended = True; break
        k = len(path) - 1
        i = next(i for i in range(k) if path[i + 1] in adj[path[0]] and path[i] in adj[path[k]])
        cyc = path[:i + 1] + path[k:i:-1]         # v0..vi vk..v(i+1), closes back to v0
        if len(cyc) == n: return cyc
        j, u = next((j, u) for j, v in enumerate(cyc) for u in adj[v] if u not in on)
        path = [u] + cyc[j:] + cyc[:j]; on.add(u)

def is_ham_cycle(c, adj):
    return len(set(c)) == len(adj) == len(c) and all(c[(i + 1) % len(c)] in adj[c[i]] for i in range(len(c)))

for n in [10, 50, 200, 400]:
    for trial in range(5):
        while True:          # random graph with minimum degree >= n/2
            adj = random_graph(n, 0.62)
            if min(len(a) for a in adj.values()) >= n / 2: break
        t0 = time.time(); c = dirac_cycle(adj); dt = time.time() - t0
        assert is_ham_cycle(c, adj)
    print(f'n={n}: Hamiltonian cycles found and checked ({dt * 1000:.1f} ms for the last one)')

In [ ]:
def hamiltonian_exact(adj):
    n = len(adj); full = (1 << n) - 1
    # reach[S] = set of v such that a path from 0 visits exactly S and ends at v (bitmask)
    reach = [0] * (1 << n); reach[1] = 1
    nb = [sum(1 << w for w in adj[v]) for v in range(n)]
    for S in range(1, 1 << n, 2):
        r = reach[S]
        while r:
            v = (r & -r).bit_length() - 1; r &= r - 1
            free = nb[v] & ~S
            while free:
                w = (free & -free).bit_length() - 1; free &= free - 1
                reach[S | 1 << w] |= 1 << w
    return any(reach[full] >> v & 1 and 0 in adj[v] for v in range(n))

def sharp_example(n):
    adj = {v: set() for v in range(n)}
    if n % 2:   # two cliques of size (n+1)/2 sharing vertex 0
        m = (n + 1) // 2; A, B = range(0, m), [0] + list(range(m, n))
    else:       # K_{n/2-1, n/2+1}
        A, B = None, None
    if A is not None:
        for X in (A, B):
            for u, v in itertools.combinations(X, 2): adj[u].add(v); adj[v].add(u)
    else:
        left = range(n // 2 - 1)
        for u in left:
            for v in range(n // 2 - 1, n): adj[u].add(v); adj[v].add(u)
    return adj
for n in range(3, 15):
    adj = sharp_example(n)
    print(n, 'min degree', min(map(len, adj.values())), '= ceil(n/2)-1:', min(map(len, adj.values())) == math.ceil(n / 2) - 1,
          ' Hamiltonian:', hamiltonian_exact(adj))

### P5.5
**How to find it.** Binary search for the insertion point: keep an interval $[\ell,r]$ of the current path with $v_\ell\to x$ and $x\to v_r$; look up the middle game and keep the half where the two ends still disagree. Counting Hamiltonian paths: the Held–Karp table counts paths by set of visited vertices and last vertex.

In [ ]:
def random_tournament(n, rng=random):
    beats = [[False] * n for _ in range(n)]
    for u, v in itertools.combinations(range(n), 2):
        if rng.random() < 0.5: beats[u][v] = True
        else: beats[v][u] = True
    return beats

def redei(beats):
    games = 0
    def b(u, v):
        nonlocal games; games += 1; return beats[u][v]
    path = [0]
    for x in range(1, len(beats)):
        if b(x, path[0]): path.insert(0, x); continue
        if b(path[-1], x): path.append(x); continue
        lo, hi = 0, len(path) - 1             # path[lo] -> x, x -> path[hi]
        while hi - lo > 1:
            mid = (lo + hi) // 2
            if b(path[mid], x): lo = mid
            else: hi = mid
        path.insert(hi, x)
    return path, games

for n in [10, 100, 1000, 3000]:
    T = random_tournament(n); p, g = redei(T)
    assert sorted(p) == list(range(n)) and all(T[a][b] for a, b in zip(p, p[1:]))
    print(f'n={n}: ranking found with {g} games looked up (n log2 n = {n * math.log2(n):.0f}, all games = {n * (n - 1) // 2})')

In [ ]:
def count_ham_paths(beats):
    n = len(beats); cnt = [[0] * n for _ in range(1 << n)]
    for v in range(n): cnt[1 << v][v] = 1
    for S in range(1, 1 << n):
        for v in range(n):
            c = cnt[S][v]
            if c:
                for w in range(n):
                    if not S >> w & 1 and beats[v][w]: cnt[S | 1 << w][w] += c
    return sum(cnt[(1 << n) - 1])

def strongly_connected(beats):
    n = len(beats)
    def reach(adj):
        seen = {0}; st = [0]
        while st:
            u = st.pop()
            for w in range(n):
                if adj(u, w) and w not in seen: seen.add(w); st.append(w)
        return len(seen) == n
    return reach(lambda u, w: beats[u][w]) and reach(lambda u, w: beats[w][u])

def ham_cycle_tournament(beats):
    n = len(beats); adj = {v: {w for w in range(n) if beats[v][w] or beats[w][v]} for v in range(n)}
    full = (1 << n) - 1; reach = [0] * (1 << n); reach[1] = 1
    for S in range(1, 1 << n, 2):
        r = reach[S]
        for v in range(n):
            if r >> v & 1:
                for w in range(n):
                    if not S >> w & 1 and beats[v][w]: reach[S | 1 << w] |= 1 << w
    return any(reach[full] >> v & 1 and beats[v][0] for v in range(n))

counts = Counter(); strong = camion = 0
for trial in range(400):
    n = random.randint(3, 9); T = random_tournament(n)
    h = count_ham_paths(T); counts[h % 2] += 1
    if strongly_connected(T):
        strong += 1; camion += ham_cycle_tournament(T)
    wins = [sum(r) for r in T]; x = wins.index(max(wins))
    assert all(y == x or T[x][y] or any(T[x][z] and T[z][y] for z in range(n)) for y in range(n))   # king
print('parity of the number of Hamiltonian paths:', dict(counts))
print(f'strongly connected tournaments: {strong}, with a Hamiltonian cycle: {camion}')
trans = [[u < v for v in range(7)] for u in range(7)]
print('transitive tournament on 7 players:', count_ham_paths(trans), 'Hamiltonian path')

Every count is odd (Rédei), every strongly connected tournament has a Hamiltonian cycle (Camion), and the player with the most wins is always a king (Landau). How large can the number of Hamiltonian paths be? Szele showed in 1943 that a random tournament has on average $n!/2^{n-1}$ of them, so some tournament has at least that many.

In [ ]:
for n in [5, 7, 9]:
    vals = [count_ham_paths(random_tournament(n)) for _ in range(300)]
    print(f'n={n}: average {np.mean(vals):.1f}, Szele n!/2^(n-1) = {math.factorial(n) / 2 ** (n - 1):.1f}, max seen {max(vals)}')

### P5.6
**How to find it.** Replace each vertex $v$ by an arc $v^-\to v^+$ of capacity $1$ and each edge $uv$ by arcs $u^+\to v^-$ and $v^+\to u^-$; add a source before $A$ and a sink after $B$. Integral flows of value $k$ are exactly $k$ disjoint $A$–$B$ paths (Chapter 6 proves max-flow min-cut in general). The brute-force side tries all vertex sets in increasing size.

In [ ]:
def max_disjoint_paths(adj, A, B):
    cap = defaultdict(int); nbr = defaultdict(set)
    def arc(u, v, c=1):
        cap[(u, v)] += c; nbr[u].add(v); nbr[v].add(u)
    for v in adj:
        arc(('in', v), ('out', v))
        for w in adj[v]: arc(('out', v), ('in', w))
    for a in A: arc('s', ('in', a))
    for b in B: arc(('out', b), 't')
    flow = 0
    while True:
        prev = {'s': None}; q = deque(['s'])
        while q and 't' not in prev:
            u = q.popleft()
            for w in nbr[u]:
                if w not in prev and cap[(u, w)] > 0: prev[w] = u; q.append(w)
        if 't' not in prev: return flow
        v = 't'
        while prev[v] is not None:
            u = prev[v]; cap[(u, v)] -= 1; cap[(v, u)] += 1; v = u
        flow += 1

def separates(adj, A, B, X):
    seen = {a for a in A if a not in X}; st = list(seen)
    while st:
        u = st.pop()
        if u in B: return False
        for w in adj[u]:
            if w not in seen and w not in X: seen.add(w); st.append(w)
    return True

def min_separator(adj, A, B):
    for k in range(len(adj) + 1):
        for X in itertools.combinations(adj, k):
            if separates(adj, A, B, set(X)): return k

random.seed(8)
for trial in range(400):
    n = random.randint(3, 11); adj = random_graph(n, random.choice([0.2, 0.35, 0.5]))
    A = set(random.sample(range(n), random.randint(1, 3))); B = set(random.sample(range(n), random.randint(1, 3)))
    assert max_disjoint_paths(adj, A, B) == min_separator(adj, A, B)
print("Menger's theorem checked on 400 random instances (A and B may overlap)")

In [ ]:
def connectivity(adj):
    # Corollary 5.9: min over non-adjacent s, t of the number of disjoint s-t paths
    vs = list(adj); best = len(vs) - 1
    for s, t in itertools.combinations(vs, 2):
        if t not in adj[s]:
            H = {v: adj[v] - {s, t} for v in adj if v not in (s, t)}     # G - s - t
            best = min(best, max_disjoint_paths(H, set(adj[s]), set(adj[t])))
    return best
def cube(d):
    return {v: {v ^ (1 << i) for i in range(d)} for v in range(1 << d)}
grid = {(i, j): {(i + a, j + b) for a, b in [(1, 0), (-1, 0), (0, 1), (0, -1)] if 0 <= i + a < 4 and 0 <= j + b < 4} for i in range(4) for j in range(4)}
for name, g in [('Petersen', pet), ('cube Q3', cube(3)), ('cube Q4', cube(4)), ('4x4 grid', grid), ('K_{3,4}', {v: (set(range(3, 7)) if v < 3 else set(range(3))) for v in range(7)})]:
    print(f'{name}: connectivity {connectivity(g)}, minimum degree {min(map(len, g.values()))}')

For each non-adjacent pair $s,t$ the code counts disjoint $N(s)$–$N(t)$ paths in $G-s-t$, exactly as in the proof of Corollary 5.9. In all these graphs the connectivity equals the minimum degree, the largest value possible, since removing the neighbours of a vertex cuts it off.

### P5.7
**How to find it.** Schmidt's chain decomposition: run a DFS; for each back edge $uv$ ($v$ an ancestor of $u$), in the order in which DFS discovered the ancestor $v$, walk from $v$ through $u$ and up the tree until reaching an already visited vertex. The first chain is a cycle, the others are ears when the graph is 2-connected. For Robbins, the DFS orientation of Exercise 5.9; check strong connectivity with two searches.

In [ ]:
def dfs_tree(adj, root=0):
    order, parent, st, seen = [], {root: None}, [(root, iter(adj[root]))], {root}
    order.append(root)
    while st:
        u, it = st[-1]
        for w in it:
            if w not in seen:
                seen.add(w); parent[w] = u; order.append(w); st.append((w, iter(adj[w]))); break
        else:
            st.pop()
    return order, parent

def ear_decomposition(adj):
    order, parent = dfs_tree(adj); pos = {v: i for i, v in enumerate(order)}
    visited, chains = set(), []
    for v in order:
        for u in adj[v]:
            if parent.get(u) != v and parent.get(v) != u and pos[u] > pos[v]:   # back edge u -> ancestor v
                chain = [v]; visited.add(v); x = u
                while x not in visited:
                    chain.append(x); visited.add(x); x = parent[x]
                chain.append(x); chains.append(chain)
    return chains

def check_ears(adj, chains):
    edges = {frozenset((a, b)) for a in adj for b in adj[a]}
    got = [frozenset(e) for c in chains for e in zip(c, c[1:])]
    if Counter(got) != Counter(edges) or chains[0][0] != chains[0][-1]: return False
    H = set(chains[0])
    for c in chains[1:]:
        if c[0] == c[-1] or c[0] not in H or c[-1] not in H or any(x in H for x in c[1:-1]): return False
        H |= set(c)
    return H == set(adj)

def two_connected(adj):
    return len(adj) >= 3 and all(connected(adj, {v}) for v in adj)

stats = Counter()
for trial in range(1000):
    n = random.randint(3, 14); adj = random_graph(n, random.choice([0.25, 0.4, 0.6]))
    if not connected(adj): continue
    tc = two_connected(adj); ok = check_ears(adj, ear_decomposition(adj)) if sum(map(len, adj.values())) else False
    assert tc == ok          # Whitney: an ear decomposition exactly when 2-connected
    stats[tc] += 1
print('connected graphs tested:', dict(stats), '(True = 2-connected, ear decomposition found and checked)')
print('Petersen graph ears:', ear_decomposition(pet))

In [ ]:
def robbins(adj):
    order, parent = dfs_tree(adj); pos = {v: i for i, v in enumerate(order)}
    arcs = set()
    for u in adj:
        for w in adj[u]:
            if parent.get(w) == u: arcs.add((u, w))              # tree edge, parent to child
            elif parent.get(u) != w and pos[u] > pos[w]: arcs.add((u, w))   # back edge, descendant to ancestor
    return arcs
def strongly_conn(vs, arcs):
    out, inn = defaultdict(set), defaultdict(set)
    for a, b in arcs: out[a].add(b); inn[b].add(a)
    def full(nb):
        s = next(iter(vs)); seen = {s}; st = [s]
        while st:
            u = st.pop()
            for w in nb[u]:
                if w not in seen: seen.add(w); st.append(w)
        return len(seen) == len(vs)
    return full(out) and full(inn)
def has_bridge(adj):
    return any(not connected({x: adj[x] - ({v} if x == u else {u} if x == v else set()) for x in adj}) for u in adj for v in adj[u])
cnt = Counter()
for trial in range(1000):
    n = random.randint(2, 15); adj = random_graph(n, random.choice([0.2, 0.35, 0.5]))
    if not connected(adj) or n < 2: continue
    arcs = robbins(adj); assert len(arcs) == sum(map(len, adj.values())) // 2
    sc = strongly_conn(set(adj), arcs)
    assert sc == (not has_bridge(adj)); cnt[sc] += 1
print("Robbins checked: the DFS orientation is strongly connected exactly when there is no bridge", dict(cnt))

def gray(n):
    return [i ^ (i >> 1) for i in range(1 << n)]
for n in range(2, 11):
    g = gray(n)
    assert len(set(g)) == 1 << n and all(bin(a ^ b).count('1') == 1 for a, b in zip(g, g[1:] + g[:1]))
print('reflected Gray codes checked up to n = 10:', [format(x, '03b') for x in gray(3)])

A last picture: how quickly does a random graph acquire a Hamiltonian cycle, compared with Dirac's guarantee? Dirac needs degree $n/2$; Pósa, and then Komlós–Szemerédi and Bollobás, showed that a random graph becomes Hamiltonian at the very moment its minimum degree reaches $2$, near $p=\frac{\ln n+\ln\ln n}{n}$, far below $\frac12$.

In [ ]:
n = 14; ps = np.linspace(0.05, 0.6, 12); frac_ham, frac_deg2 = [], []
for p in ps:
    h = d = 0
    for _ in range(60):
        adj = random_graph(n, p)
        h += hamiltonian_exact(adj); d += min(map(len, adj.values())) >= 2
    frac_ham.append(h / 60); frac_deg2.append(d / 60)
plt.figure(figsize=(6, 3))
plt.plot(ps, frac_ham, 'o-', label='Hamiltonian')
plt.plot(ps, frac_deg2, 's--', label='minimum degree $\\geq 2$')
plt.axvline(0.5, color='gray', lw=0.8); plt.text(0.505, 0.1, 'Dirac needs\nall degrees $\\geq n/2$', fontsize=8)
plt.xlabel('edge probability p'); plt.title(f'random graphs on {n} vertices'); plt.legend(); plt.tight_layout(); plt.show()